# 02 — Consequential scenarios

**Audience:** Users building prospective consequential inventories from an ecoinvent consequential database.

**Prerequisites:** A consequential ecoinvent database (3.8 or newer), its biosphere, a valid IAM key, and familiarity with the quickstart notebook.

**Learning goals:** configure the consequential system model, control marginal-supplier identification, and export a consequential scenario database.


## Outline

1. Validate the consequential source database.
2. Configure marginal-mix arguments.
3. Apply selected transformations.
4. Export and review assumptions.


In [ ]:
import os

import bw2data as bd

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-consequential"
SOURCE_DATABASE = "ecoinvent-3.12-consequential"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
PREMISE_KEY = os.environ.get("PREMISE_KEY")

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")

bd.projects.set_current(PROJECT)
missing = [
    name for name in (SOURCE_DATABASE, BIOSPHERE_DATABASE) if name not in bd.databases
]
if missing:
    raise ValueError(f"Missing Brightway databases: {missing}")


## 1. Configure marginal mixes

The reference configuration models a short, myopic demand change using a market-average lead time. `range time` and `duration` cannot both be non-zero.


In [ ]:
SYSTEM_ARGS = {
    "range time": 2,
    "duration": 0,
    "foresight": False,
    "lead time": False,
    "capital replacement rate": True,
    "measurement": 0,
    "weighted slope start": 0.75,
    "weighted slope end": 1.0,
}

SCENARIOS = [
    {"model": "remind", "pathway": "SSP2-NDC", "year": 2030},
]


## 2. Build the consequential database

`system_model="consequential"` must agree with the source database. Electricity and steel are sufficient for a focused tutorial run; use `ndb.update()` for every sector.


In [ ]:
ndb = NewDatabase(
    scenarios=[scenario.copy() for scenario in SCENARIOS],
    source_db=SOURCE_DATABASE,
    source_version="3.12",
    source_type="brightway",
    system_model="consequential",
    system_args=SYSTEM_ARGS,
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)
ndb.update(["electricity", "steel"])


In [ ]:
OUTPUT_DATABASE = "premise-remind-ssp2-ndc-2030-consequential"
ndb.write_db_to_brightway(name=OUTPUT_DATABASE)
assert OUTPUT_DATABASE in bd.databases


## Interpretation and pitfalls

- Premise constructs marginal market mixes for IAM-linked markets and excludes constrained suppliers using its consequential data files.
- Changing foresight, lead-time treatment, or the measurement method changes the modeled decision context; document these choices with results.
- Use `validate_consequential_marginal_mixes.py` in this directory to inspect the argument families and equations independently.

## Exercise

Compare the reference assumptions with perfect foresight. Create a new argument dictionary without mutating `SYSTEM_ARGS`.


In [ ]:
perfect_foresight_args = {
    **SYSTEM_ARGS,
    "foresight": True,
}
perfect_foresight_args
